In [ ]:
import pandas as pd, numpy as np, json
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_parquet("data/processed/flow_ready.parquet")
FEATS = json.load(open("data/processed/features.json"))

train = df[df["split"] == "train"]
val = df[df["split"] == "val"]


benign = train[train["Label"] == "BENIGN"].sample(300_000, random_state=42)
train_bal = pd.concat([benign, train[train["Label"] != "BENIGN"]])
print(train_bal["Label"].value_counts())

rf = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                            n_jobs=-1, random_state=42)
rf.fit(train_bal[FEATS], train_bal["Label"])

pred = rf.predict(val[FEATS])
print(classification_report(val["Label"], pred, digits=3, zero_division=0))


imp = pd.Series(rf.feature_importances_, index=FEATS).sort_values(ascending=False)
print(imp.head(10))

Label
BENIGN                        300000
DoS Hulk                      103628
DDoS                           76807
DoS GoldenEye                   6172
FTP-Patator                     3559
DoS slowloris                   3230
DoS Slowhttptest                3137
SSH-Patator                     1932
PortScan                        1110
Web Attack - Brute Force         882
Bot                              848
Web Attack - XSS                 392
Infiltration                      22
Web Attack - Sql Injection        13
Heartbleed                         7
Name: count, dtype: int64
                            precision    recall  f1-score   support

                    BENIGN      0.999     0.998     0.998    379365
                       Bot      0.218     0.254     0.235       283
                      DDoS      0.997     1.000     0.999     25602
             DoS GoldenEye      0.992     0.966     0.979      2057
                  DoS Hulk      0.990     0.999     0.995     34543
    

In [2]:
import joblib
joblib.dump(rf, "models/rf_baseline.joblib")

# save class probabilities on val (fuzzy layer input later)
proba = pd.DataFrame(rf.predict_proba(val[FEATS]), columns=rf.classes_, index=val.index)
proba.to_parquet("data/processed/rf_val_proba.parquet")
print("saved")

saved
